In [1]:
from pathlib import Path
import sys
import pandas as pd

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import load_run, save_labeled_run
from src.labels import load_label_config, discover_labeled_runs, label_run_sensors, validate_label_transitions, plot_labeled_sensors, save_labeled_plot

In [2]:
# Discover all runs with their label configurations
raw_data_dir = PROJECT_ROOT / "data" / "raw"
runs_with_labels = discover_labeled_runs(raw_data_dir)

print(f"Found {len(runs_with_labels)} run(s) with label config(s):")
for run_dir, config_path in runs_with_labels.items():
    print(f"  - {run_dir.name} -> {config_path.relative_to(PROJECT_ROOT)}")


Found 2 run(s) with label config(s):
  - log_20260223_142511.490 -> data/raw/test_2/log_20260223_142511.490/labels_config.json
  - log_20260226_102148.990 -> data/raw/test_3/log_20260226_102148.990/labels_config.json


In [3]:
# Load all runs with their label configurations
all_runs = {}

for run_dir, config_path in runs_with_labels.items():
    label_configs = load_label_config(config_path)
    run = load_run(run_dir, include_pose=True)
    
    all_runs[run.run_id] = {
        'run': run,
        'label_configs': label_configs,
        'config_path': config_path
    }
    
    print(f"\n{run.run_id}:")
    print(f"  Label config: {config_path.relative_to(PROJECT_ROOT)}")
    print(f"  Labels: {', '.join(label_configs.keys())}")
    print(f"  Samples: ACC={len(run.acc)}, GYRO={len(run.gyro)}, ODO={len(run.odo)}, POSE={len(run.pose) if run.pose is not None else 'N/A'}")



log_20260223_142511.490:
  Label config: data/raw/test_2/log_20260223_142511.490/labels_config.json
  Labels: grass, smooth_terrain, muddy_dirt_track
  Samples: ACC=243359, GYRO=243378, ODO=365121, POSE=584131

log_20260226_102148.990:
  Label config: data/raw/test_3/log_20260226_102148.990/labels_config.json
  Labels: grass, smooth_terrain, muddy_dirt_track
  Samples: ACC=181067, GYRO=182779, ODO=272900, POSE=439816


In [4]:
# Apply labels to all sensors for each run
labeled_runs = {}

for run_id, run_data in all_runs.items():
    run = run_data['run']
    label_configs = run_data['label_configs']
    
    # Label all sensors
    labeled_sensors = label_run_sensors(run, label_configs)
    labeled_runs[run_id] = labeled_sensors
    
    print(f"\n{run_id.upper()} - Label Distribution:")
    for sensor_name, labeled_df in labeled_sensors.items():
        label_counts = labeled_df['label'].value_counts(dropna=False)
        print(f"  {sensor_name}: {dict(label_counts)}")



LOG_20260223_142511.490 - Label Distribution:
  acc: {nan: np.int64(206528), 'smooth_terrain': np.int64(15583), 'muddy_dirt_track': np.int64(13666), 'grass': np.int64(7582)}
  gyro: {nan: np.int64(206547), 'smooth_terrain': np.int64(15584), 'muddy_dirt_track': np.int64(13665), 'grass': np.int64(7582)}
  odo: {nan: np.int64(309879), 'smooth_terrain': np.int64(23374), 'muddy_dirt_track': np.int64(20496), 'grass': np.int64(11372)}
  pose: {nan: np.int64(495744), 'smooth_terrain': np.int64(37399), 'muddy_dirt_track': np.int64(32794), 'grass': np.int64(18194)}

LOG_20260226_102148.990 - Label Distribution:
  acc: {nan: np.int64(166067), 'grass': np.int64(5000), 'smooth_terrain': np.int64(5000), 'muddy_dirt_track': np.int64(5000)}
  gyro: {nan: np.int64(167779), 'grass': np.int64(5000), 'smooth_terrain': np.int64(5000), 'muddy_dirt_track': np.int64(5000)}
  odo: {nan: np.int64(250400), 'grass': np.int64(7500), 'smooth_terrain': np.int64(7500), 'muddy_dirt_track': np.int64(7500)}
  pose: {na

In [5]:
# Validate label assignments
for run_id, labeled_sensors in labeled_runs.items():
    print(f"\n{'='*60}")
    print(f"Validation: {run_id}")
    print('='*60)
    
    # Validate transitions in accelerometer data
    acc_labeled = labeled_sensors['acc']
    validation = validate_label_transitions(acc_labeled, max_transitions=3)
    
    print(f"Total transitions: {validation['total_transitions']}")
    for transition in validation['transition_samples']:
        print(f"\nTransition {transition['transition_num']} (row {transition['row_idx']}):")
        print(transition['samples'])



Validation: log_20260223_142511.490
Total transitions: 206530

Transition 1 (row 1):
              t label
0  1.771853e+09   NaN
1  1.771853e+09   NaN
2  1.771853e+09   NaN
3  1.771853e+09   NaN

Transition 2 (row 2):
              t label
0  1.771853e+09   NaN
1  1.771853e+09   NaN
2  1.771853e+09   NaN
3  1.771853e+09   NaN
4  1.771853e+09   NaN

Transition 3 (row 3):
              t label
1  1.771853e+09   NaN
2  1.771853e+09   NaN
3  1.771853e+09   NaN
4  1.771853e+09   NaN
5  1.771853e+09   NaN

Validation: log_20260226_102148.990
Total transitions: 166069

Transition 1 (row 1):
              t label
0  1.772098e+09   NaN
1  1.772098e+09   NaN
2  1.772098e+09   NaN
3  1.772098e+09   NaN

Transition 2 (row 2):
              t label
0  1.772098e+09   NaN
1  1.772098e+09   NaN
2  1.772098e+09   NaN
3  1.772098e+09   NaN
4  1.772098e+09   NaN

Transition 3 (row 3):
              t label
1  1.772098e+09   NaN
2  1.772098e+09   NaN
3  1.772098e+09   NaN
4  1.772098e+09   NaN
5  1.77209

In [6]:
# Save all labeled runs
from dataclasses import replace

output_root = PROJECT_ROOT / "data" / "labeled"

for run_id, run_data in all_runs.items():
    run = run_data['run']
    labeled_sensors = labeled_runs[run_id]
    
    # Create labeled RunData
    run_labeled = replace(
        run,
        acc=labeled_sensors['acc'],
        gyro=labeled_sensors['gyro'],
        odo=labeled_sensors['odo'],
        pose=labeled_sensors.get('pose')
    )
    
    # Save to disk
    save_labeled_run(run_labeled, output_root)
    
    print(f"\nSaved {run_id} to {output_root / run_id}")
    for f in sorted((output_root / run_id).glob("*.csv")):
        file_lines = len(pd.read_csv(f))
        print(f"  {f.name}: {file_lines} rows")



Saved log_20260223_142511.490 to /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/labeled/log_20260223_142511.490
  log_t0_acc_1.csv: 243359 rows


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_49525/2302610568.py:24: DtypeWarning: Columns (0: label) have mixed types. Specify dtype option on import or set low_memory=False.
  file_lines = len(pd.read_csv(f))


  log_t0_encoder_velocity.csv: 365121 rows
  log_t0_gyro_1.csv: 243378 rows
  log_t0_pose.csv: 584131 rows

Saved log_20260226_102148.990 to /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/labeled/log_20260226_102148.990
  log_t0_acc_1.csv: 181067 rows
  log_t0_encoder_velocity.csv: 272900 rows


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_49525/2302610568.py:24: DtypeWarning: Columns (0: label) have mixed types. Specify dtype option on import or set low_memory=False.
  file_lines = len(pd.read_csv(f))


  log_t0_gyro_1.csv: 182779 rows
  log_t0_pose.csv: 439816 rows


In [7]:
# Generate and save labeled plots for all runs
import matplotlib.pyplot as plt

plots_output_root = PROJECT_ROOT / "reports" / "labeled"

for run_id, run_data in all_runs.items():
    labeled_sensors = labeled_runs[run_id]
    
    # Create labeled plot
    fig = plot_labeled_sensors(
        acc=labeled_sensors['acc'],
        gyro=labeled_sensors['gyro'],
        odo=labeled_sensors['odo'],
        tcol='t_rel',
        show=False
    )
    
    # Save plot to reports/labeled/{run_id}/
    plot_path = save_labeled_plot(
        fig,
        run_id=run_id,
        output_root=plots_output_root,
        filename="labeled_sensors_plot.png",
        dpi=150
    )
    
    plt.close(fig)  # Close figure to free memory
    
    print(f"\nSaved plot for {run_id}:")
    print(f"  {plot_path.relative_to(PROJECT_ROOT)}")


Saved plot for log_20260223_142511.490:
  reports/labeled/log_20260223_142511.490/labeled_sensors_plot.png

Saved plot for log_20260226_102148.990:
  reports/labeled/log_20260226_102148.990/labeled_sensors_plot.png
